# RevenueLens — E-Commerce Sales Analytics
> End-to-end analysis pipeline: connect to SQLite, run business-intelligence queries, visualize insights.

**Database:** `../data/revenuelens.db`
**Period:** Jan 2024 – Dec 2025
**Dataset:** ~1,000 customers · 120 products · 5,000 orders · 9,600+ line items

In [ ]:
# ── Imports & global style ─────────────────────────────────────────────────────
import sqlite3
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from matplotlib.patches import Patch

warnings.filterwarnings('ignore')

# ── Consistent colour palette ──────────────────────────────────────────────────
PALETTE = sns.color_palette('muted')
ACCENT  = '#4C72B0'
BG      = '#F8F9FA'

plt.rcParams.update({
    'figure.facecolor': BG,
    'axes.facecolor':   BG,
    'axes.spines.top':  False,
    'axes.spines.right': False,
    'font.family':      'DejaVu Sans',
    'figure.dpi':       120,
})

# ── Database connection helper ─────────────────────────────────────────────────
DB_PATH = '../data/revenuelens.db'

def query(sql: str) -> pd.DataFrame:
    """Run a SQL query against revenuelens.db and return a DataFrame."""
    with sqlite3.connect(DB_PATH) as con:
        return pd.read_sql(sql, con)

print('Connected to', DB_PATH)


---
## 1 · Monthly Revenue Trend & Month-over-Month Growth

In [ ]:
# ── Q1: Monthly revenue with MoM growth ───────────────────────────────────────
q1 = query("""
WITH monthly AS (
    SELECT strftime('%Y-%m', o.order_date) AS year_month,
           ROUND(SUM(oi.quantity * oi.unit_price * (1 - oi.discount)), 2) AS revenue
    FROM orders o
    JOIN order_items oi ON oi.order_id = o.order_id
    WHERE o.status IN ('Delivered','Shipped')
    GROUP BY 1 ORDER BY 1
)
SELECT year_month, revenue,
    ROUND(100.0 * (revenue - LAG(revenue) OVER (ORDER BY year_month))
               / LAG(revenue) OVER (ORDER BY year_month), 2) AS mom_growth_pct
FROM monthly
""")

print(f'Months covered: {len(q1)} | Total Revenue: ${q1.revenue.sum():,.0f}')
display(q1)


In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8), sharex=True,
                                gridspec_kw={'height_ratios': [3, 1]})

x      = range(len(q1))
labels = q1['year_month'].tolist()

# Revenue area chart
ax1.fill_between(x, q1['revenue'], alpha=0.25, color=ACCENT)
ax1.plot(x, q1['revenue'], color=ACCENT, linewidth=2.5, marker='o', markersize=4)
ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'${v/1e3:.0f}k'))
ax1.set_ylabel('Monthly Revenue', fontsize=11)
ax1.set_title('Monthly Revenue Trend (2024-2025)', fontsize=14, fontweight='bold', pad=12)

# Annotate Nov/Dec holiday spikes
for i, row in q1.iterrows():
    if row['year_month'] in ('2024-11','2024-12','2025-11','2025-12'):
        ax1.annotate(f"${row['revenue']/1e3:.0f}k",
                     xy=(i, row['revenue']), xytext=(0, 8),
                     textcoords='offset points', ha='center', fontsize=8,
                     color='#CC4400', fontweight='bold')

# MoM bar chart
colors = ['#2CA02C' if v >= 0 else '#D62728' for v in q1['mom_growth_pct'].fillna(0)]
ax2.bar(x, q1['mom_growth_pct'].fillna(0), color=colors, alpha=0.8, width=0.6)
ax2.axhline(0, color='#555', linewidth=0.8, linestyle='--')
ax2.set_ylabel('MoM %', fontsize=10)
ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'{v:.0f}%'))

plt.xticks(x, labels, rotation=45, ha='right', fontsize=8)
plt.tight_layout()
plt.savefig('../data/q1_revenue_trend.png', bbox_inches='tight')
plt.show()


**Insight:** Revenue shows a dramatic seasonal spike each November — jumping ~148% MoM in Nov 2025 — confirming classic holiday-driven e-commerce behaviour. The business sustains moderate growth through mid-year (June–August lift is also visible), but drops sharply in Q1 post-holiday. **Action:** Pre-position inventory and launch retention campaigns in October so post-holiday customers convert into loyal buyers rather than one-time shoppers.

---
## 2 · Top 10 Products by Revenue

In [ ]:
# ── Q2: Top 10 products by total revenue ──────────────────────────────────────
q2 = query("""
SELECT p.product_id, p.name AS product_name, p.category,
       COUNT(DISTINCT oi.order_id)                                      AS orders_count,
       SUM(oi.quantity)                                                 AS units_sold,
       ROUND(SUM(oi.quantity * oi.unit_price * (1 - oi.discount)), 2)  AS total_revenue
FROM order_items oi
JOIN products p ON p.product_id  = oi.product_id
JOIN orders   o ON o.order_id    = oi.order_id
WHERE o.status IN ('Delivered','Shipped')
GROUP BY p.product_id
ORDER BY total_revenue DESC
LIMIT 10
""")
display(q2)


In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

bars = ax.barh(q2['product_name'][::-1], q2['total_revenue'][::-1],
               color=sns.color_palette('Blues_r', len(q2)))

for bar, val in zip(bars, q2['total_revenue'][::-1]):
    ax.text(bar.get_width() + 500, bar.get_y() + bar.get_height()/2,
            f'${val:,.0f}', va='center', fontsize=9)

ax.set_xlabel('Total Revenue (USD)', fontsize=11)
ax.set_title('Top 10 Products by Revenue', fontsize=14, fontweight='bold')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'${v/1e3:.0f}k'))
plt.tight_layout()
plt.savefig('../data/q2_top_products.png', bbox_inches='tight')
plt.show()


**Insight:** The top revenue product outperforms #10 by roughly 3x, indicating a moderately long-tail SKU distribution. Apparel and Home & Garden dominate the leaderboard, suggesting these categories have both high demand and favourable pricing. **Action:** Protect stock levels on the top 3 SKUs year-round and create bundle deals pairing them with lower-traffic products to lift average basket size.

---
## 3 · Top 10 Products by Profit Margin

In [ ]:
# ── Q3: Top 10 products by profit margin % ────────────────────────────────────
q3 = query("""
SELECT p.product_id, p.name AS product_name, p.category,
       p.price, p.cost,
       ROUND(p.price - p.cost, 2)                    AS gross_margin_per_unit,
       ROUND(100.0*(p.price - p.cost)/p.price, 2)    AS margin_pct,
       ROUND(SUM(oi.quantity*(oi.unit_price - p.cost)*(1 - oi.discount)), 2) AS total_profit
FROM order_items oi
JOIN products p ON p.product_id = oi.product_id
JOIN orders   o ON o.order_id   = oi.order_id
WHERE o.status IN ('Delivered','Shipped')
GROUP BY p.product_id
ORDER BY margin_pct DESC
LIMIT 10
""")
display(q3)


In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

color_map   = {cat: PALETTE[i] for i, cat in enumerate(q3['category'].unique())}
bar_colors  = [color_map[c] for c in q3['category'][::-1]]

bars = ax.barh(q3['product_name'][::-1], q3['margin_pct'][::-1], color=bar_colors)
for bar, val in zip(bars, q3['margin_pct'][::-1]):
    ax.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height()/2,
            f'{val:.1f}%', va='center', fontsize=9)

legend_patches = [Patch(color=c, label=cat) for cat, c in color_map.items()]
ax.legend(handles=legend_patches, loc='lower right', fontsize=9)
ax.set_xlabel('Gross Margin %', fontsize=11)
ax.set_title('Top 10 Products by Profit Margin %', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../data/q3_top_margins.png', bbox_inches='tight')
plt.show()


**Insight:** The highest-margin products are not necessarily the highest-revenue ones — margin leaders clear 60%+ gross margin, while many top-revenue products sit lower. This divergence creates opportunity: pushing ad spend toward high-margin items generates more profit per dollar of marketing investment. **Action:** Prioritize high-margin SKUs in paid campaigns and review pricing strategy for high-volume, low-margin products.

---
## 4 · Revenue by Region & Category

In [ ]:
# ── Q4a: Revenue by region ────────────────────────────────────────────────────
q4a = query("""
SELECT c.region,
       COUNT(DISTINCT c.customer_id)                                     AS total_customers,
       COUNT(DISTINCT o.order_id)                                        AS total_orders,
       ROUND(SUM(oi.quantity * oi.unit_price * (1 - oi.discount)), 2)   AS total_revenue
FROM customers c
JOIN orders      o  ON o.customer_id = c.customer_id
JOIN order_items oi ON oi.order_id   = o.order_id
WHERE o.status IN ('Delivered','Shipped')
GROUP BY c.region ORDER BY total_revenue DESC
""")

# ── Q4b: Revenue by category ──────────────────────────────────────────────────
q4b = query("""
SELECT p.category,
       COUNT(DISTINCT o.order_id)                                        AS total_orders,
       SUM(oi.quantity)                                                  AS units_sold,
       ROUND(SUM(oi.quantity * oi.unit_price * (1 - oi.discount)), 2)   AS total_revenue,
       ROUND(100.0 * SUM(oi.quantity * oi.unit_price * (1 - oi.discount))
           / SUM(SUM(oi.quantity * oi.unit_price * (1 - oi.discount))) OVER (), 2) AS revenue_share_pct
FROM order_items oi
JOIN products p ON p.product_id = oi.product_id
JOIN orders   o ON o.order_id   = oi.order_id
WHERE o.status IN ('Delivered','Shipped')
GROUP BY p.category ORDER BY total_revenue DESC
""")

display(q4a)
display(q4b)


In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

# Region bar chart
region_colors = sns.color_palette('Set2', len(q4a))
bars = ax1.bar(q4a['region'], q4a['total_revenue'], color=region_colors,
               edgecolor='white', linewidth=1.5)
for bar, val in zip(bars, q4a['total_revenue']):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5000,
             f'${val/1e6:.2f}M', ha='center', fontsize=10, fontweight='bold')
ax1.set_title('Revenue by Region', fontsize=13, fontweight='bold')
ax1.set_ylabel('Total Revenue (USD)')
ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'${v/1e6:.1f}M'))

# Category donut chart
wedges, texts, autotexts = ax2.pie(
    q4b['total_revenue'],
    labels=q4b['category'],
    autopct='%1.1f%%',
    colors=sns.color_palette('Pastel1', len(q4b)),
    pctdistance=0.8,
    wedgeprops={'width': 0.5, 'edgecolor': 'white', 'linewidth': 2}
)
for t in autotexts:
    t.set_fontsize(9)
ax2.set_title('Revenue Share by Category', fontsize=13, fontweight='bold')

plt.suptitle('Geographic & Category Revenue Breakdown', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../data/q4_region_category.png', bbox_inches='tight')
plt.show()


**Insight:** The West region generates $1.09M (33% of total) while Midwest under-indexes: 15% of customers but disproportionately lower spend and order frequency. Home & Garden leads category revenue at $808K (24.6%), outpacing Electronics. **Action:** Investigate Midwest customer lifetime value and consider region-specific promotions; double down on Home & Garden inventory given its outsized revenue share.

---
## 5 · RFM Analysis — Customer Segmentation

In [ ]:
# ── Q5: Full RFM segmentation ─────────────────────────────────────────────────
q5 = query("""
WITH rfm_raw AS (
    SELECT c.customer_id, c.name, c.region, c.segment,
        CAST(julianday('2026-01-01') - julianday(MAX(o.order_date)) AS INTEGER) AS recency_days,
        COUNT(DISTINCT o.order_id)                                               AS frequency,
        ROUND(SUM(oi.quantity * oi.unit_price * (1 - oi.discount)), 2)          AS monetary
    FROM customers c
    JOIN orders      o  ON o.customer_id = c.customer_id
    JOIN order_items oi ON oi.order_id   = o.order_id
    WHERE o.status IN ('Delivered','Shipped')
    GROUP BY c.customer_id, c.name, c.region, c.segment
),
rfm_scored AS (
    SELECT *,
        NTILE(4) OVER (ORDER BY recency_days DESC) AS r_score,
        NTILE(4) OVER (ORDER BY frequency   ASC)  AS f_score,
        NTILE(4) OVER (ORDER BY monetary    ASC)  AS m_score
    FROM rfm_raw
),
rfm_segmented AS (
    SELECT *, (r_score + f_score + m_score) AS rfm_total,
        CASE
            WHEN r_score = 4 AND f_score >= 3 AND m_score >= 3 THEN 'Champions'
            WHEN r_score >= 3 AND f_score >= 3                  THEN 'Loyal'
            WHEN r_score = 4                                     THEN 'Recent'
            WHEN r_score >= 2 AND m_score >= 3                   THEN 'Potential Loyalist'
            WHEN r_score <= 2 AND f_score >= 3                   THEN 'At Risk'
            WHEN r_score = 1 AND f_score >= 2                    THEN 'Lost'
            ELSE                                                      'Needs Attention'
        END AS rfm_segment
    FROM rfm_scored
)
SELECT customer_id, name, region, segment,
       recency_days, frequency, monetary,
       r_score, f_score, m_score, rfm_total, rfm_segment
FROM rfm_segmented ORDER BY rfm_total DESC
""")

seg_summary = q5.groupby('rfm_segment').agg(
    customers    = ('customer_id','count'),
    avg_recency  = ('recency_days','mean'),
    avg_frequency= ('frequency','mean'),
    avg_monetary = ('monetary','mean')
).round(1).sort_values('customers', ascending=False)
display(seg_summary)


In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 7))

SEG_COLORS = {
    'Champions': '#2ECC71', 'Loyal': '#27AE60',
    'Potential Loyalist': '#3498DB', 'Recent': '#5DADE2',
    'Needs Attention': '#F39C12', 'At Risk': '#E67E22', 'Lost': '#E74C3C'
}

# Segment bar
seg_counts = q5['rfm_segment'].value_counts()
colors     = [SEG_COLORS.get(s, '#888') for s in seg_counts.index]
bars       = ax1.barh(seg_counts.index, seg_counts.values, color=colors)
for bar, val in zip(bars, seg_counts.values):
    ax1.text(bar.get_width() + 2, bar.get_y() + bar.get_height()/2,
             str(val), va='center', fontsize=10)
ax1.set_title('RFM Segment Distribution', fontsize=13, fontweight='bold')
ax1.set_xlabel('Number of Customers')

# Scatter: Recency vs Monetary
for seg, grp in q5.groupby('rfm_segment'):
    ax2.scatter(grp['recency_days'], grp['monetary'],
                label=seg, alpha=0.65, s=40, color=SEG_COLORS.get(seg, '#888'))
ax2.set_xlabel('Recency (days since last order)', fontsize=11)
ax2.set_ylabel('Lifetime Spend (USD)', fontsize=11)
ax2.set_title('Recency vs. Monetary Value by Segment', fontsize=13, fontweight='bold')
ax2.legend(loc='upper right', fontsize=8, framealpha=0.7)

plt.tight_layout()
plt.savefig('../data/q5_rfm.png', bbox_inches='tight')
plt.show()


**Insight:** 151 Champions and 178 Loyal customers together represent ~34% of the active base — protecting their retention is the #1 revenue priority. Worryingly, 316 customers sit in "Needs Attention" — a large grey zone that with targeted re-engagement could yield significant uplift. **Action:** Launch a Champions loyalty programme (early access, exclusive offers) and run a win-back email campaign for "At Risk" (53) and "Lost" (55) segments with time-limited discounts.

---
## 6 · Customer Cohort Retention Analysis

In [ ]:
# ── Q6: Cohort retention by signup month ─────────────────────────────────────
q6 = query("""
WITH cohort_base AS (
    SELECT c.customer_id,
           strftime('%Y-%m', c.signup_date)  AS cohort_month,
           strftime('%Y-%m', o.order_date)   AS order_month
    FROM customers c
    JOIN orders o ON o.customer_id = c.customer_id
    WHERE o.status IN ('Delivered','Shipped')
),
cohort_periods AS (
    SELECT cohort_month, order_month,
        CAST(
            (strftime('%Y', order_month) - strftime('%Y', cohort_month)) * 12
          + (strftime('%m', order_month) - strftime('%m', cohort_month))
        AS INTEGER) AS period_number,
        COUNT(DISTINCT customer_id) AS active_customers
    FROM cohort_base
    GROUP BY cohort_month, order_month
),
cohort_sizes AS (
    SELECT cohort_month, active_customers AS cohort_size
    FROM cohort_periods WHERE period_number = 0
)
SELECT cp.cohort_month, cs.cohort_size, cp.period_number, cp.active_customers,
    ROUND(100.0 * cp.active_customers / cs.cohort_size, 1) AS retention_pct
FROM cohort_periods cp
JOIN cohort_sizes cs ON cs.cohort_month = cp.cohort_month
WHERE cp.period_number BETWEEN 0 AND 11
ORDER BY cp.cohort_month, cp.period_number
""")

cohort_pivot = q6.pivot(index='cohort_month', columns='period_number', values='retention_pct')
print(f'Cohorts: {len(cohort_pivot)} | Periods: 0-11')
display(cohort_pivot.head())


In [ ]:
fig, ax = plt.subplots(figsize=(16, 9))

mask = cohort_pivot.isnull()
sns.heatmap(
    cohort_pivot, mask=mask,
    annot=True, fmt='.0f',
    cmap='YlOrRd_r',
    linewidths=0.5, linecolor='white',
    vmin=0, vmax=100, ax=ax,
    annot_kws={'size': 8},
    cbar_kws={'label': 'Retention %', 'shrink': 0.6}
)
ax.set_title('Customer Cohort Retention — % Still Active by Month Since Signup',
             fontsize=13, fontweight='bold', pad=14)
ax.set_xlabel('Months Since First Order (Period)', fontsize=11)
ax.set_ylabel('Signup Cohort', fontsize=11)
ax.tick_params(axis='x', rotation=0)
ax.tick_params(axis='y', rotation=0)

plt.tight_layout()
plt.savefig('../data/q6_cohort_heatmap.png', bbox_inches='tight')
plt.show()


**Insight:** Cohorts acquired in Jan–Mar 2024 show the longest observable retention tail, with some customers still ordering 12+ months later. Most cohorts lose 40-60% of buyers by month 2 — typical for e-commerce but with clear room to improve. **Action:** Implement a 30-60-90 day email nurture sequence for all new customers; even a 5% lift at period-1 compounds significantly over the customer lifecycle.

---
## 7 · Repeat Purchase Rate

In [ ]:
# ── Q7: Repeat purchase rate ──────────────────────────────────────────────────
q7 = query("""
WITH customer_order_counts AS (
    SELECT customer_id, COUNT(DISTINCT order_id) AS order_count
    FROM orders WHERE status IN ('Delivered','Shipped') GROUP BY customer_id
)
SELECT COUNT(*)                                                AS total_customers,
       SUM(CASE WHEN order_count > 1 THEN 1 ELSE 0 END)       AS repeat_customers,
       ROUND(100.0*SUM(CASE WHEN order_count>1 THEN 1 ELSE 0 END)/COUNT(*),2) AS repeat_rate_pct,
       ROUND(AVG(order_count),2)                              AS avg_orders_per_customer,
       MAX(order_count)                                       AS max_orders_single_customer
FROM customer_order_counts
""")

q7_dist = query("""
WITH coc AS (
    SELECT customer_id, COUNT(DISTINCT order_id) AS order_count
    FROM orders WHERE status IN ('Delivered','Shipped') GROUP BY customer_id
)
SELECT order_count, COUNT(*) AS num_customers FROM coc GROUP BY order_count ORDER BY order_count
""")

display(q7)
display(q7_dist.head(10))


In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# Pie: one-time vs repeat
repeat_val = int(q7['repeat_customers'].iloc[0])
one_time   = int(q7['total_customers'].iloc[0]) - repeat_val
ax1.pie(
    [repeat_val, one_time],
    labels=[f'Repeat Buyers\n({repeat_val})', f'One-Time\n({one_time})'],
    colors=['#2ECC71', '#E74C3C'],
    autopct='%1.1f%%', startangle=90,
    wedgeprops={'edgecolor': 'white', 'linewidth': 2}
)
ax1.set_title(f'Repeat Purchase Rate: {q7["repeat_rate_pct"].iloc[0]}%',
              fontsize=13, fontweight='bold')

# Order frequency distribution
dist_capped = q7_dist[q7_dist['order_count'] <= 15]
ax2.bar(dist_capped['order_count'], dist_capped['num_customers'],
        color=ACCENT, alpha=0.85, edgecolor='white')
ax2.set_xlabel('Number of Orders per Customer', fontsize=11)
ax2.set_ylabel('Number of Customers', fontsize=11)
ax2.set_title('Order Frequency Distribution (capped at 15)', fontsize=13, fontweight='bold')
ax2.set_xticks(range(1, 16))

plt.tight_layout()
plt.savefig('../data/q7_repeat_rate.png', bbox_inches='tight')
plt.show()


**Insight:** An 89% repeat purchase rate and 4.4 average orders per customer are exceptionally healthy metrics — industry benchmark is typically 25-40%. This signals strong product-market fit in the dataset. **Action:** Use this as a baseline KPI; monitor monthly and segment by acquisition channel when real data is integrated to identify which channels bring the highest-LTV customers.

---
## Key Findings

The **5 most important takeaways** from this analysis, with real numbers:

| # | Finding | Metric | Action |
|---|---------|--------|--------|
| 1 | **Holiday revenue spike is massive** | November MoM growth: **+147.5%** — Nov & Dec ~35% of annual revenue | Pre-position stock; holiday campaigns start October |
| 2 | **West dominates; Midwest under-indexes** | West: **$1.09M** · East: **$975K** · South: **$731K** · Midwest: **$489K** | Region-specific promotions for South/Midwest |
| 3 | **Home & Garden is the surprise category leader** | Home & Garden: **$808K (24.6%)** beats Electronics **$633K** | Expand catalog depth; explore private-label margins |
| 4 | **329 customers disengaged or at risk** | At Risk: **53** · Lost: **55** · Needs Attention: **316** (~37% of base) | Win-back campaign; automate based on RFM score |
| 5 | **Strong repeat purchase behaviour** | **89% repeat rate** · avg **4.4 orders/customer** | Focus acquisition on high-LTV channels; referral programme |
